# KuaiRand Data Understanding and EDA

This notebook uses PySpark/Spark SQL for scalable exploration. The intended flow is:

1. Inspect raw CSV files and schemas.
2. Convert CSV to Parquet in the bronze layer.
3. Use Parquet for row counts, missing values, duplicates, timestamps, interaction flags, and basic user/item/temporal statistics.
4. Prepare reusable columns for later session analysis and preference-drift analysis.

In [ ]:
from pathlib import Path
import importlib
import os
import sys

os.environ.setdefault("SPARK_LOCAL_IP", "127.0.0.1")
os.environ.setdefault("PYSPARK_SUBMIT_ARGS", "--driver-memory 4g pyspark-shell")
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from pyspark.sql import SparkSession, functions as F
from pyspark.sql import Window

import recommender.data.kuairand as kuairand_module
import recommender.spark as spark_module

kuairand_module = importlib.reload(kuairand_module)
spark_module = importlib.reload(spark_module)
read_csv = kuairand_module.read_csv
get_spark = spark_module.get_spark

try:
    spark = get_spark("kuairand-eda", reset=True)
except TypeError:
    # Fallback for an already-running kernel that still has an older helper loaded.
    spark = (
        SparkSession.builder.master("local[*]")
        .appName("kuairand-eda")
        .config("spark.sql.session.timeZone", "UTC")
        .config("spark.sql.shuffle.partitions", "8")
        .config("spark.driver.bindAddress", "127.0.0.1")
        .config("spark.driver.host", "127.0.0.1")
        .config("spark.local.dir", str(PROJECT_ROOT / "data/spark-tmp"))
        .config("spark.sql.execution.arrow.pyspark.enabled", "true")
        .config("spark.pyspark.python", sys.executable)
        .config("spark.pyspark.driver.python", sys.executable)
        .getOrCreate()
    )

spark.sparkContext.setLogLevel("ERROR")
print(f"Python executable: {sys.executable}")
print(f"Spark {spark.version} | master={spark.sparkContext.master} | driver={spark.sparkContext.getConf().get('spark.driver.host')}")
spark


## Configure Paths

The notebook auto-detects raw CSV files in `data/raw/kuairand`, then `data/raw`, then `data`. Parquet outputs go to `data/bronze/kuairand`.

If you changed Spark memory or local IP settings after the first cell already ran, restart the kernel before continuing. Spark only applies those settings when the JVM starts.

In [ ]:
RAW_CSV_CANDIDATES = [
    PROJECT_ROOT / "data/raw/kuairand",
    PROJECT_ROOT / "data/raw",
    PROJECT_ROOT / "data",
]
RAW_CSV_DIR = next((p for p in RAW_CSV_CANDIDATES if list(p.glob("*.csv"))), RAW_CSV_CANDIDATES[0])
BRONZE_PARQUET_DIR = PROJECT_ROOT / "data/bronze/kuairand"

print(f"Raw CSV directory: {RAW_CSV_DIR}")
print(f"Bronze Parquet directory: {BRONZE_PARQUET_DIR}")

USER_COL = "user_id"
ITEM_COL = "video_id"
DATE_COL = "date"
HOURMIN_COL = "hourmin"
TIME_MS_COL = "time_ms"

INTERACTION_COLS = [
    "is_click",
    "is_like",
    "is_follow",
    "is_comment",
    "is_forward",
    "is_hate",
    "long_view",
    "is_profile_enter",
]


## Inspect Raw Files

In [ ]:
csv_files = sorted(RAW_CSV_DIR.glob("*.csv"))
if not csv_files:
    raise FileNotFoundError(f"No CSV files found in {RAW_CSV_DIR}. Put KuaiRand CSVs under data/raw or data/raw/kuairand.")
[(p.name, round(p.stat().st_size / 1024 / 1024, 3)) for p in csv_files]


In [ ]:
raw_tables = {}
for path in csv_files:
    name = path.stem
    df = read_csv(spark, path)
    raw_tables[name] = df
    print(f"\n{name}")
    df.printSchema()
    df.show(5, truncate=False)


## Convert CSV to Parquet

Parquet is the default format for analysis because it is columnar, typed, compressed, and much faster for repeated Spark scans than CSV.

In [ ]:
BRONZE_PARQUET_DIR.mkdir(parents=True, exist_ok=True)

for path in csv_files:
    name = path.stem
    target = BRONZE_PARQUET_DIR / name
    print(f"converting {path.name} -> {target}")
    df = read_csv(spark, path)
    df.write.mode("overwrite").parquet(str(target))
    print(f"wrote {target}")

spark.catalog.clearCache()


In [ ]:
parquet_tables = {}
for path in sorted(p for p in BRONZE_PARQUET_DIR.iterdir() if p.is_dir()):
    parquet_tables[path.name] = spark.read.parquet(str(path))

list(parquet_tables.keys())

## Understand Each KuaiRand Table

Before session or preference analysis, inspect each data type on its own: interaction logs, user features, video metadata, and video historical statistics.

In [ ]:
table_groups = {
    "interaction_logs": sorted([name for name in parquet_tables if name.startswith("log_")]),
    "user_features": sorted([name for name in parquet_tables if name.startswith("user_features")]),
    "video_basic": sorted([name for name in parquet_tables if name.startswith("video_features_basic")]),
    "video_statistics": sorted([name for name in parquet_tables if name.startswith("video_features_statistic")]),
}

for group_name, names in table_groups.items():
    print(f"{group_name}: {names}")


In [ ]:
table_profile_rows = []
for group_name, names in table_groups.items():
    for name in names:
        df = parquet_tables[name]
        table_profile_rows.append((group_name, name, df.count(), len(df.columns), ", ".join(df.columns[:8])))

spark.createDataFrame(
    table_profile_rows,
    ["group", "table", "rows", "columns", "first_columns"],
).orderBy("group", "table").show(truncate=False)


In [ ]:
def inspect_table(name, n=5):
    df = parquet_tables[name]
    print(f"\n{name}")
    df.printSchema()
    df.show(n, truncate=False)

for names in table_groups.values():
    for name in names:
        inspect_table(name, n=3)


### Missing Values by Table

In [ ]:
def missing_profile(df, table_name):
    total = df.count()
    exprs = [F.sum(F.col(c).isNull().cast("long")).alias(c) for c in df.columns]
    row = df.agg(*exprs).collect()[0].asDict()
    rows = [(table_name, col, missing, missing / total if total else None) for col, missing in row.items() if missing]
    if not rows:
        rows = [(table_name, "<none>", 0, 0.0)]
    return rows

missing_rows = []
for names in table_groups.values():
    for name in names:
        missing_rows.extend(missing_profile(parquet_tables[name], name))

spark.createDataFrame(missing_rows, ["table", "column", "missing", "missing_rate"]).orderBy(
    "table", F.desc("missing")
).show(100, truncate=False)


### User Feature Tables

In [ ]:
for name in table_groups["user_features"]:
    df = parquet_tables[name]
    print(f"\n{name}: users={df.select(USER_COL).distinct().count()}")
    df.select(
        "follow_user_num", "fans_user_num", "friend_user_num", "register_days"
    ).summary("count", "min", "25%", "50%", "75%", "max").show(truncate=False)
    for col in [
        "user_active_degree", "is_lowactive_period", "is_live_streamer", "is_video_author",
        "follow_user_num_range", "fans_user_num_range", "friend_user_num_range", "register_days_range",
    ]:
        print(f"\nTop values for {col}")
        df.groupBy(col).count().orderBy(F.desc("count")).show(20, truncate=False)


### Video Basic Feature Tables

In [ ]:
for name in table_groups["video_basic"]:
    df = parquet_tables[name]
    video_count = df.select(ITEM_COL).distinct().count()
    author_count = df.select("author_id").distinct().count()
    print(f"\n{name}: videos={video_count}, authors={author_count}")
    df.select("video_duration", "server_width", "server_height").summary(
        "count", "min", "25%", "50%", "75%", "max"
    ).show(truncate=False)
    for col in ["video_type", "upload_type", "visible_status", "music_type", "tag"]:
        print(f"\nTop values for {col}")
        df.groupBy(col).count().orderBy(F.desc("count")).show(20, truncate=False)


### Video Historical Statistic Tables

In [ ]:
video_stat_focus_cols = [
    "counts", "show_cnt", "show_user_num", "play_cnt", "play_user_num", "play_duration",
    "valid_play_cnt", "long_time_play_cnt", "short_time_play_cnt", "play_progress",
    "like_cnt", "comment_cnt", "follow_cnt", "share_cnt", "collect_cnt", "report_cnt",
]

for name in table_groups["video_statistics"]:
    df = parquet_tables[name]
    existing_cols = [c for c in video_stat_focus_cols if c in df.columns]
    print(f"\n{name}: videos={df.select(ITEM_COL).distinct().count()}")
    df.select(*existing_cols).summary("count", "min", "25%", "50%", "75%", "max").show(truncate=False)
    print("\nTop videos by play_cnt")
    df.select(ITEM_COL, "play_cnt", "play_user_num", "play_duration", "like_cnt", "comment_cnt", "share_cnt").orderBy(
        F.desc("play_cnt")
    ).show(20, truncate=False)


### Interaction Log Tables

In [ ]:
for name in table_groups["interaction_logs"]:
    df = parquet_tables[name]
    print(f"\n{name}")
    df.agg(
        F.count("*").alias("rows"),
        F.countDistinct(USER_COL).alias("users"),
        F.countDistinct(ITEM_COL).alias("videos"),
        F.min(DATE_COL).alias("min_date"),
        F.max(DATE_COL).alias("max_date"),
        *[F.sum(F.col(c).cast("long")).alias(c) for c in INTERACTION_COLS if c in df.columns],
    ).show(truncate=False)
    for col in ["is_rand", "tab"]:
        if col in df.columns:
            print(f"\nDistribution for {col}")
            df.groupBy(col).count().orderBy(col).show(truncate=False)


## Pick Interaction Tables

KuaiRand has log tables plus user/video feature tables. This cell unions all log-like tables with matching columns for interaction-level analysis.

In [ ]:
log_names = [name for name, df in parquet_tables.items() if {USER_COL, ITEM_COL}.issubset(df.columns) and name.startswith("log_")]
log_names

In [ ]:
logs = None
for name in log_names:
    df = parquet_tables[name].withColumn("source_table", F.lit(name))
    logs = df if logs is None else logs.unionByName(df, allowMissingColumns=True)

logs.printSchema()
logs.show(5, truncate=False)

## Row Counts, Users, Items

In [ ]:
table_counts = []
for name, df in parquet_tables.items():
    table_counts.append((name, df.count(), len(df.columns)))

spark.createDataFrame(table_counts, ["table", "rows", "columns"]).orderBy("table").show(truncate=False)

In [ ]:
logs.agg(
    F.count("*").alias("interactions"),
    F.countDistinct(USER_COL).alias("users"),
    F.countDistinct(ITEM_COL).alias("items"),
    F.countDistinct("source_table").alias("log_tables"),
).show(truncate=False)

## Missing Values and Duplicates

In [ ]:
def missing_summary(df):
    exprs = [F.sum(F.col(c).isNull().cast("long")).alias(c) for c in df.columns]
    return df.agg(*exprs)

missing_summary(logs).show(vertical=True, truncate=False)

In [ ]:
duplicate_keys = [USER_COL, ITEM_COL, DATE_COL, HOURMIN_COL, TIME_MS_COL]
available_duplicate_keys = [c for c in duplicate_keys if c in logs.columns]

# Exact duplicate checks across the full interaction log are shuffle-heavy.
# For EDA, inspect duplicate candidates on a deterministic sample first.
DUPLICATE_SAMPLE_FRACTION = 0.05
logs_sample = logs.sample(withReplacement=False, fraction=DUPLICATE_SAMPLE_FRACTION, seed=42)

logs_sample.groupBy(*available_duplicate_keys).count().where(F.col("count") > 1).orderBy(
    F.desc("count")
).show(20, truncate=False)


## Timestamp and Interaction Types

In [ ]:
def with_event_time(df):
    out = df
    if DATE_COL in out.columns:
        out = out.withColumn("event_date", F.to_date(F.col(DATE_COL).cast("string"), "yyyyMMdd"))
    if TIME_MS_COL in out.columns:
        out = out.withColumn("event_ts", F.to_timestamp(F.from_unixtime((F.col(TIME_MS_COL) / 1000).cast("long"))))
    if HOURMIN_COL in out.columns:
        out = out.withColumn("hourmin_str", F.lpad(F.col(HOURMIN_COL).cast("string"), 4, "0"))
        out = out.withColumn("event_hour", F.substring("hourmin_str", 1, 2).cast("int"))
        out = out.withColumn("event_minute", F.substring("hourmin_str", 3, 2).cast("int"))
    return out

logs_ts = with_event_time(logs)
logs_ts.select(DATE_COL, HOURMIN_COL, TIME_MS_COL, "event_date", "event_ts", "event_hour", "event_minute").show(10, truncate=False)

In [ ]:
logs_ts.agg(
    F.min("event_date").alias("min_date"),
    F.max("event_date").alias("max_date"),
    F.min("event_ts").alias("min_ts"),
    F.max("event_ts").alias("max_ts"),
).show(truncate=False)

In [ ]:
existing_interaction_cols = [c for c in INTERACTION_COLS if c in logs_ts.columns]
interaction_summary = logs_ts.agg(*[F.sum(F.col(c).cast("long")).alias(c) for c in existing_interaction_cols])
interaction_summary.show(truncate=False)

## Basic User, Item, and Temporal Statistics

In [ ]:
user_stats = logs_ts.groupBy(USER_COL).agg(
    F.count("*").alias("interactions"),
    F.countDistinct(ITEM_COL).alias("distinct_items"),
    *[F.sum(F.col(c).cast("long")).alias(c) for c in existing_interaction_cols],
)

user_stats.orderBy(F.desc("interactions")).show(20, truncate=False)

In [ ]:
item_stats = logs_ts.groupBy(ITEM_COL).agg(
    F.count("*").alias("interactions"),
    F.countDistinct(USER_COL).alias("distinct_users"),
    *[F.sum(F.col(c).cast("long")).alias(c) for c in existing_interaction_cols],
)

item_stats.orderBy(F.desc("interactions")).show(20, truncate=False)

In [ ]:
logs_ts.groupBy("event_date").agg(
    F.count("*").alias("interactions"),
    F.countDistinct(USER_COL).alias("users"),
    F.countDistinct(ITEM_COL).alias("items"),
).orderBy("event_date").show(60, truncate=False)

In [ ]:
logs_ts.groupBy("event_hour").agg(
    F.count("*").alias("interactions"),
    F.countDistinct(USER_COL).alias("users"),
).orderBy("event_hour").show(24, truncate=False)

## Session and Preference-Drift Preparation

These columns are not final features yet. They prepare the interaction log for later sessionization and short-term preference-drift analysis.

In [ ]:
SESSION_GAP_MINUTES = 30

# Session analysis should focus on meaningful watch/click events, not every impression row.
# This keeps the session view interpretable and avoids huge local window shuffles.
session_base = logs_ts.where(
    (F.col("is_click") == 1)
    | (F.col("long_view") == 1)
    | (F.col("play_time_ms") > 0)
)

w_user_time = Window.partitionBy(USER_COL).orderBy("event_ts")

logs_sequence = (
    session_base
    .withColumn("prev_event_ts", F.lag("event_ts").over(w_user_time))
    .withColumn("gap_seconds", F.col("event_ts").cast("long") - F.col("prev_event_ts").cast("long"))
    .withColumn(
        "is_new_session",
        F.when(F.col("prev_event_ts").isNull(), F.lit(1))
        .when(F.col("gap_seconds") > SESSION_GAP_MINUTES * 60, F.lit(1))
        .otherwise(F.lit(0)),
    )
    .withColumn("session_index", F.sum("is_new_session").over(w_user_time.rowsBetween(Window.unboundedPreceding, 0)))
    .withColumn("session_id", F.concat_ws("_", F.col(USER_COL).cast("string"), F.col("session_index").cast("string")))
)

session_base.agg(F.count("*").alias("session_candidate_events")).show(truncate=False)
logs_sequence.select(USER_COL, ITEM_COL, "event_ts", "gap_seconds", "session_id", *existing_interaction_cols).show(20, truncate=False)


In [ ]:
session_stats = logs_sequence.groupBy("session_id", USER_COL).agg(
    F.min("event_ts").alias("session_start"),
    F.max("event_ts").alias("session_end"),
    F.count("*").alias("events"),
    F.countDistinct(ITEM_COL).alias("distinct_items"),
    *[F.sum(F.col(c).cast("long")).alias(c) for c in existing_interaction_cols],
)

session_stats.orderBy(F.desc("events")).show(20, truncate=False)

In [ ]:
daily_user_pref = logs_ts.groupBy(USER_COL, "event_date").agg(
    F.count("*").alias("interactions"),
    F.countDistinct(ITEM_COL).alias("distinct_items"),
    *[F.avg(F.col(c).cast("double")).alias(f"{c}_rate") for c in existing_interaction_cols],
)

daily_user_pref.orderBy(F.desc("interactions"), USER_COL, "event_date").show(20, truncate=False)